In [ ]:
import sisl 
import numpy as np
import matplotlib.pyplot as plt
from sisl.viz.processors.math import normalize

from utils.loader import load_datastructure, path_finder, parse_lwc
from utils.plots import plot_with_center, mark_electrode

project_dir = path_finder("QTM")


# List all structures

In [ ]:
!ls ../results -l | grep L

## Extract all bandgaps and Rs

In [ ]:
results_dir = path_finder("QTM") / "results"
list_of_lwc = [parse_lwc(folder) for folder in sorted(results_dir.glob("L*_W*_C*"))]
modulations = np.unique([str(m.split("_")[1]) for m in load_datastructure(list_of_lwc[0])[-1].keys() if m.startswith("bg")]).tolist()
print("Found modulations :", modulations)

# initialize storage dictionary
all_bgs = {mod: [] for mod in modulations}
all_bgs["R"] = []
all_bgs["W"] = []
# loop over all LWC structures and collect bandgaps for different modulations and outer radius R
for LWC in list_of_lwc:
    if LWC[0] != 1:
        continue  # only process L=1 structures for now
    DATA = load_datastructure(LWC)[-1]
    print(f"Processing LWC: L{LWC[0]} W{LWC[1]:02} C{LWC[2]:02} with outer radius R = {DATA['R']} Å")
    for mod in modulations:
        bg_key = f"bg_{mod}"
        all_bgs[mod].append(DATA[bg_key])
    all_bgs["R"].append(DATA["R"])
    all_bgs["W"].append(LWC[1])
# convert lists to numpy arrays and squeeze
for key in all_bgs.keys():
    all_bgs[key] = np.array(all_bgs[key]).squeeze()

## Plot bandgaps for a given modulation vs R

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6,4), sharey=True, dpi=200)
idxs = np.argsort(all_bgs["W"], stable=True)
x1 = all_bgs["W"][idxs]
x2 = all_bgs["R"][idxs]
# Create the second x-axis on top
ax_top = ax.twiny()

# Set the limits of the top axis to match the bottom 
# (This assumes R and W scale linearly with each other)

# first plot: bandgap vs outer radius for different modulations
for mod in modulations:
    y = all_bgs[mod][idxs]
    ax.plot(x1, y, marker='d', label=f"None" if mod == "C0" else f"{float(mod[1:])}", alpha=0.7)


ax_top.set_xlim(ax.get_xlim())

# Map the ticks from R values to W values
# We pick a few indices to show as labels
tick_locations = x1[::len(x1)//5] # Pick 5 points
tick_labels = x2[::len(x2)//5]    # Get corresponding R values

ax_top.set_xticks(tick_locations)
ax_top.set_xticklabels([f"{float(R):.2f}" for R in tick_labels])
ax.set_xticks(tick_locations)
# ax.set_xticklabels([f"{(W)}" for W in tick_locations])    
ax.set_xlabel("Width $W$ [#]", fontsize=14)
ax_top.set_xlabel("Radius [Å]", fontsize=14)
ax.set_ylabel("$E-E_f$ [eV]", fontsize=14)
ax.legend()
ax.grid()

ax.set_xticklabels(ax.get_xticklabels(), fontsize=10)
ax_top.set_xticklabels(ax_top.get_xticklabels(), fontsize=10)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=10)
# ax_top.set_xlabel("Width $W$ [#]", fontsize=14)
fig.suptitle("Band gap for different Modulations", fontsize=16)
plt.tight_layout()
fig.savefig(project_dir / "presentation" / "bg_vs_R_W.png", dpi=200)

In [ ]:
for i in range(len(x1)):
    print(f"{i:>2}: W = {x1[i]:>2}, R = {x2[i]:.3f} Å")

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))
mid = 6
bgs = all_bgs[modulations[mid]]
rs = all_bgs["R"]
ax.plot(rs, bgs, marker='o', linestyle='None')
ax.set_title(f"Bandgaps for {modulations[mid]}", fontsize=16)

In [ ]:
x,y = np.vstack((bgs, rs))
fig, ax = plt.subplots(figsize=(8,6))
ax.scatter(rs, bgs, marker='o')
ax.set_xlabel("Outer Radius R (Å)",fontsize=14)
ax.set_ylabel("Bandgap (eV)", fontsize=14)
ax.set_title("Bandgap vs Outer Radius", fontsize=16)

In [ ]:
def f(x):
    return x, None
a = f(0); a

In [ ]:
%load_ext line_profiler
from utils.energies import multi_LDOS, diagonal_of_inverse, lr_energies
from utils.structure import find_nearest_atoms

In [ ]:
i = -1
electrode, ribbon, device, data = load_datastructure(list_of_lwc[i]); list_of_lwc[i]
device.na

In [ ]:
lprun_args = dict(device=device, 
                  electrode=electrode, 
                  energies=np.linspace(-2, 2, num=2),
                  lr_indices=data["lr_idx"],
                  Nk=1,
                  form="csc",
                  eta=1e-5,
                  modulate_SE=None,
                  LDOS_E0=None,
                #   sites=None
                  sites=find_nearest_atoms(device.xyz, device.center(), neighbours=6)
)
%lprun -f lr_energies -f multi_LDOS  multi_LDOS(**lprun_args)